# RIS Interview Prep — Python, Polars, Machine Learning & Quant Finance Statistics

A working reference notebook covering:
1. General Python logic and syntax
2. Polars (vs. pandas) fundamentals
3. Machine learning for anomaly/outlier detection — Isolation Forest, a single-layer
   neural network, and other unsupervised methods
4. Statistics as applied to quantitative finance
5. Core factors used in quantitative/factor investing

Each section leads with the likely interview framing, then runnable code, then a
short **Why** note explaining the reasoning — the part worth being able to say out
loud, not just the syntax.


In [ ]:
import numpy as np
import pandas as pd

try:
    import polars as pl
    HAS_POLARS = True
except ImportError:
    HAS_POLARS = False
    print("polars not installed — pip install polars")

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)


## Section 1 — General Python Logic

Warmup / sanity-check territory: data structures, comprehensions, error handling, decorators.

**Q: Deduplicate a list while preserving order.**

**Why:** set lookup is O(1) avg vs. O(n) for `if item not in result` — matters at scale.


In [ ]:
def dedupe_preserve_order(items: list) -> list:
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

dedupe_preserve_order(["A1", "A2", "A1", "A3", "A2"])


**Q: List comprehension vs. generator expression — when would you use one over the other?**

**Why:** generators matter when processing large datasets where you don't need everything
in memory at once — same "why lazy evaluation" reasoning that shows up again with Polars.


In [ ]:
squares_list = [x**2 for x in range(10)]   # materializes full list in memory
squares_gen = (x**2 for x in range(10))    # lazy, one value at a time
list(squares_gen)


**Q: What's the bug in a mutable default argument, and how do you fix it?**

**Why:** default arguments are evaluated once at function definition, not per call —
a classic "gotcha" that tests real fluency, not just syntax memorization.


In [ ]:
def bad_default(item, bucket=[]):      # BUG: shared list across calls
    bucket.append(item)
    return bucket

def good_default(item, bucket=None):   # FIX: new list each call
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(bad_default("x"), bad_default("y"))   # bucket persists across calls — bug
print(good_default("x"), good_default("y")) # fresh each time — correct


**Q: How would you handle an error in a pipeline stage without silently swallowing it?**

**Why:** this ties directly to a "zero manual review" pipeline philosophy — errors need
to surface with context, not disappear silently behind a bare `except: pass`.


In [ ]:
def process_stage(value):
    try:
        return 100 / value
    except ZeroDivisionError as e:
        raise ValueError(f"Invalid input value=0 in process_stage: {e}") from e

try:
    process_stage(0)
except ValueError as e:
    print("Caught with context:", e)


**Q: Explain a decorator and give a use case relevant to a data pipeline.**

**Why:** decorators let you add cross-cutting concerns (timing, logging, validation) to
every pipeline stage without repeating code — directly relevant to a multi-stage pipeline.


In [ ]:
import time, functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.5f}s")
        return result
    return wrapper

@timer
def slow_computation(n):
    return sum(i**2 for i in range(n))

slow_computation(500_000)


## Section 2 — Polars (vs. pandas)

Polars questions tend to be conceptual ("why is it faster") rather than pure syntax
recall — the reasoning below is what's worth being able to explain, not just the code.


**Sample data used throughout this section.**


In [ ]:
sample_pd = pd.DataFrame({
    "security_id": ["A1", "A2", "A3", "A4", "A5", "A6"],
    "sector": ["Tech", "Tech", "Financials", "Financials", "Energy", "Energy"],
    "market_cap": [500, 300, 800, 200, 150, 900],
    "pe_ratio": [25.0, np.nan, 15.0, 18.0, 40.0, 12.0],
    "return_pct": [0.05, -0.02, 0.01, 0.03, -0.10, 0.07],
})

if HAS_POLARS:
    sample_pl = pl.DataFrame({
        "security_id": ["A1", "A2", "A3", "A4", "A5", "A6"],
        "sector": ["Tech", "Tech", "Financials", "Financials", "Energy", "Energy"],
        "market_cap": [500, 300, 800, 200, 150, 900],
        "pe_ratio": [25.0, None, 15.0, 18.0, 40.0, 12.0],
        "return_pct": [0.05, -0.02, 0.01, 0.03, -0.10, 0.07],
    })

sample_pd


**Q: Why is Polars considered faster than pandas — show it, don't just say it.**

**Why:** `.lazy()` lets Polars build a query plan and optimize it (predicate/projection
pushdown) *before* executing — it can skip unread columns and push filters down to the
scan itself. pandas executes each line eagerly with no cross-line optimization.


In [ ]:
if HAS_POLARS:
    eager_result = sample_pl.filter(pl.col("market_cap") > 200).select(["security_id", "market_cap"])

    lazy_result = (
        sample_pl.lazy()
        .filter(pl.col("market_cap") > 200)
        .select(["security_id", "market_cap"])
        .collect()
    )
    print(lazy_result)


**Q: How would you compute a sector-neutral z-score in pandas vs. Polars?**

**Why:** pandas uses `groupby().transform()` to broadcast a group aggregate back to
original row shape. Polars' `.over("sector")` does the same thing but composes inline
inside a single expression chain, which is part of what makes lazy-plan optimization possible.


In [ ]:
# pandas
sample_pd["sector_zscore_pd"] = sample_pd.groupby("sector")["pe_ratio"].transform(
    lambda x: (x - x.mean()) / x.std()
)

if HAS_POLARS:
    sample_pl = sample_pl.with_columns(
        ((pl.col("pe_ratio") - pl.col("pe_ratio").mean().over("sector"))
         / pl.col("pe_ratio").std().over("sector")).alias("sector_zscore_pl")
    )

sample_pd[["security_id", "sector", "sector_zscore_pd"]]


**Q: Why avoid `.apply(axis=1)` on large DataFrames? Show the vectorized fix.**

**Why:** `.apply(axis=1)` loops in Python under the hood and forfeits vectorization;
`.clip()` (or `where`/`np.select`) operates on the whole column at once at the C level.


In [ ]:
# anti-pattern — slow row-by-row
sample_pd["capped_slow"] = sample_pd.apply(lambda row: min(row["return_pct"], 0.05), axis=1)

# vectorized — fast
sample_pd["capped_fast"] = sample_pd["return_pct"].clip(upper=0.05)

sample_pd[["security_id", "return_pct", "capped_fast"]]


**Q: When would you still reach for pandas instead of Polars?**

Small datasets where performance doesn't matter, heavy reliance on a pandas-only
library/ecosystem integration, or team/codebase consistency mid-migration — not
everything needs to move at once.


## Section 3 — Machine Learning: Outlier & Anomaly Detection

Directly relevant to a "zero manual review" pipeline where confidence scoring is the
control mechanism. Focus: Isolation Forest, a single-layer neural network (autoencoder-
style), and other common unsupervised outlier-detection methods.


**Synthetic dataset:** mostly "normal" points clustered together, with a handful of
injected outliers — mirrors a factor score distribution with a few bad data points.


In [ ]:
n_normal = 200
n_outliers = 10

normal_data = np.random.normal(loc=0, scale=1, size=(n_normal, 2))
outlier_data = np.random.uniform(low=-6, high=6, size=(n_outliers, 2))
X = np.vstack([normal_data, outlier_data])
true_labels = np.array([0]*n_normal + [1]*n_outliers)   # 1 = actual outlier, for reference only

X.shape


### Isolation Forest

**Core logic:** randomly partitions the data with random splits on random features.
Outliers are "easier to isolate" — they end up separated into their own partition in
fewer splits than normal points, since they sit in sparse regions of the feature space.
The anomaly score is based on the average path length (number of splits) needed to
isolate each point across many random trees — short average path length = more anomalous.

**Key hyperparameter:** `contamination` — the expected proportion of outliers in the
data. This is usually the first thing to check if the model is over- or under-flagging.

**Good for:** globally extreme points, works well in higher dimensions, doesn't assume
any particular distribution shape.


In [ ]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X)

iso_preds = iso_forest.predict(X)          # -1 = anomaly, 1 = normal
iso_scores = iso_forest.decision_function(X)  # higher = more normal, lower = more anomalous

print("Flagged as anomaly:", (iso_preds == -1).sum(), "of", len(X))


**Q: Isolation Forest is flagging 15% of securities as anomalies when you expect 1-2%.
What's your diagnostic path?**

1. Check the `contamination` parameter first — cheap, mechanical, most likely culprit.
2. Check for distributional shift/drift in the input data — did an upstream feature change scale or range?
3. Sample the actually-flagged points directly and eyeball whether they look genuinely
   anomalous or just borderline.

**Watch-out:** don't reach for "overfitting" here — that concept doesn't map cleanly onto
an unsupervised model with no labeled ground truth to overfit against.


### Single-Layer Neural Network (Autoencoder-style, for reconstruction-error anomaly detection)

**Core logic:** train a small network to reconstruct its own input through a
lower-dimensional bottleneck. Points that are "normal" get reconstructed well (low
error); points that are structurally different from the training distribution get
reconstructed poorly (high error) — reconstruction error becomes the anomaly score.

A single hidden layer keeps this close to a linear compression (similar in spirit to PCA),
which is a reasonable "simplest version" to describe if asked to build one from scratch.


In [ ]:
class SingleLayerAutoencoder:
    """Minimal single-hidden-layer autoencoder using NumPy only —
    illustrates the mechanics without a deep learning framework."""

    def __init__(self, input_dim, hidden_dim, lr=0.01):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, input_dim) * 0.1
        self.b2 = np.zeros(input_dim)
        self.lr = lr

    def forward(self, X):
        hidden = np.tanh(X @ self.W1 + self.b1)     # encode
        output = hidden @ self.W2 + self.b2          # decode
        return hidden, output

    def train_step(self, X):
        hidden, output = self.forward(X)
        error = output - X
        loss = np.mean(error ** 2)

        # backprop (simplified, batch gradient descent)
        d_output = 2 * error / X.shape[0]
        dW2 = hidden.T @ d_output
        db2 = d_output.sum(axis=0)
        d_hidden = (d_output @ self.W2.T) * (1 - hidden ** 2)
        dW1 = X.T @ d_hidden
        db1 = d_hidden.sum(axis=0)

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        return loss

    def reconstruction_error(self, X):
        _, output = self.forward(X)
        return np.mean((output - X) ** 2, axis=1)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ae = SingleLayerAutoencoder(input_dim=2, hidden_dim=1, lr=0.05)
for epoch in range(300):
    loss = ae.train_step(X_scaled)

recon_errors = ae.reconstruction_error(X_scaled)
threshold = np.percentile(recon_errors, 95)   # flag top 5% as anomalies
ae_flags = recon_errors > threshold

print(f"Final training loss: {loss:.4f}")
print("Flagged as anomaly:", ae_flags.sum(), "of", len(X))


**Q: Isolation Forest vs. autoencoder — how would you choose between them?**

- Isolation Forest: cheaper, faster to train, no gradient tuning, works well for
  globally extreme / sparse-region outliers. Good default first choice.
- Autoencoder: better for complex, non-linear anomalies where "abnormal" isn't simply
  "far from everything else" but rather "doesn't fit the learned structure" — costs more
  to train and tune (architecture, learning rate, epochs).

**Q: How do you convert reconstruction error into a usable confidence score?**

Normalize/scale the reconstruction error (e.g., percentile rank or min-max scale within
a rolling window) and invert it — low error → high confidence, high error → low
confidence — then apply the same kind of threshold-based flag as any other confidence
scoring mechanism.

**Q: PCA vs. autoencoder — what's the relationship?**

A single-layer linear autoencoder (no non-linear activation) is mathematically very
close to PCA — both learn a lower-dimensional linear projection that minimizes
reconstruction error. The autoencoder's advantage only shows up once you add
non-linear activations and more layers, which let it capture non-linear structure PCA can't.


### Other Unsupervised Outlier-Detection Methods (breadth, in case asked "what else could you use")


**Local Outlier Factor (LOF)**
Compares a point's local density to the density of its neighbors — flags points that
sit in a notably sparser neighborhood than their neighbors, even if they aren't globally
extreme. Good for outliers that are only anomalous *relative to a local cluster*
(e.g., a sector-specific value that's fine market-wide but strange within its sector).


In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_preds = lof.fit_predict(X)   # -1 = anomaly, 1 = normal
print("LOF flagged as anomaly:", (lof_preds == -1).sum(), "of", len(X))


**Elliptic Envelope (Robust Covariance)**
Assumes the "normal" data roughly follows a Gaussian distribution and fits a robust
ellipse around the bulk of the data; points far outside it are flagged. Simple and fast,
but the Gaussian assumption is a real limitation for skewed financial data (e.g., returns,
which aren't symmetric or normally distributed in the tails).


In [ ]:
envelope = EllipticEnvelope(contamination=0.05, random_state=42)
env_preds = envelope.fit_predict(X)
print("Elliptic Envelope flagged as anomaly:", (env_preds == -1).sum(), "of", len(X))


**PCA-based reconstruction error**
Project data down to fewer components, then reconstruct back up — points with large
reconstruction error don't fit the dominant patterns in the data. This is the direct
"cheaper linear cousin" of the autoencoder approach above.


In [ ]:
pca = PCA(n_components=1)
X_reduced = pca.fit_transform(X_scaled)
X_reconstructed = pca.inverse_transform(X_reduced)
pca_recon_error = np.mean((X_scaled - X_reconstructed) ** 2, axis=1)

pca_threshold = np.percentile(pca_recon_error, 95)
pca_flags = pca_recon_error > pca_threshold
print("PCA-based flagged as anomaly:", pca_flags.sum(), "of", len(X))


**Quick comparison, if asked "which would you pick and why":**

| Method | Assumes distribution shape? | Local vs. global outliers | Cost |
|---|---|---|---|
| Isolation Forest | No | Global | Low |
| Autoencoder | No | Global + non-linear structure | Medium-High |
| LOF | No | Local (density-based) | Medium |
| Elliptic Envelope | Yes (Gaussian) | Global | Low |
| PCA reconstruction | Linear structure only | Global | Low |

For financial fundamentals data (skewed, fat-tailed), Isolation Forest or LOF are
generally safer defaults than Elliptic Envelope, precisely because they don't assume
a Gaussian shape.


## Section 4 — Statistics in the Context of Quantitative Finance

Core statistical concepts as they actually get used in a factor/index research and
production context.


### Z-scores and standardization

**Core logic:** `(value - mean) / std_dev` — expresses a value as "how many standard
deviations from the mean." In factor construction, this is almost always computed
**within a group** (e.g., within sector), not across the whole universe, so you're
measuring relative standing among peers rather than picking up sector-level valuation
differences.


In [ ]:
factor_df = pd.DataFrame({
    "security_id": [f"S{i}" for i in range(1, 11)],
    "sector": ["Tech"]*5 + ["Financials"]*5,
    "value_metric": [12, 15, 9, 22, 18, 30, 28, 25, 33, 20],
})

factor_df["sector_zscore"] = factor_df.groupby("sector")["value_metric"].transform(
    lambda x: (x - x.mean()) / x.std()
)
factor_df


### Beta and Alpha (CAPM building blocks)

**Beta:** `Cov(stock, market) / Var(market)` — sensitivity of a stock's returns to
market returns. Beta > 1 means more volatile than the market; beta < 1 means less.

**Alpha:** `Mean(stock returns) - beta * Mean(market returns)` — the return left over
after removing what's explained by market exposure. This is the return a factor or
strategy is actually earning *beyond* simple market beta.


In [ ]:
np.random.seed(1)
market_returns = np.random.normal(0.001, 0.01, 100)
stock_returns = 1.6 * market_returns + np.random.normal(0.0004, 0.005, 100)  # true beta ~1.6

beta = np.cov(stock_returns, market_returns)[0, 1] / np.var(market_returns)
alpha = np.mean(stock_returns) - beta * np.mean(market_returns)

print(f"Estimated beta: {beta:.3f}")
print(f"Estimated alpha (per period): {alpha:.5f}")


### Hypothesis testing on factor returns

**Core logic:** null hypothesis = the factor's true average return is zero (no real
effect). A t-test checks whether the observed average return is unlikely enough under
that null to reject it. The p-value is the probability of seeing data this extreme
*if the null were true* — not "the probability the null is true."


In [ ]:
from scipy import stats

factor_returns = np.random.normal(0.002, 0.01, 60)   # simulate a factor's period returns

t_stat, p_value = stats.ttest_1samp(factor_returns, popmean=0)
print(f"t-statistic: {t_stat:.3f}, p-value: {p_value:.4f}")
print("Reject null (factor return != 0)?" , p_value < 0.05)


### Fama-MacBeth regression (conceptual + simplified illustration)

**Core logic:** two-step procedure for testing factor models across many securities
over time.
1. **Step 1:** at each time period, run a cross-sectional regression of returns
   against factor exposures — gives one factor-return estimate per period.
2. **Step 2:** average those period-by-period estimates and test whether the average
   is significantly different from zero.

**Why it exists instead of one pooled regression:** returns across securities in the
same period are correlated with each other — a pooled regression wrongly treats every
observation as independent, understating true uncertainty.


In [ ]:
n_periods = 24
n_securities = 30

period_betas = []
for t in range(n_periods):
    exposures = np.random.normal(0, 1, n_securities)          # factor exposure per security
    true_premium = 0.003
    returns = true_premium * exposures + np.random.normal(0, 0.02, n_securities)

    # Step 1: cross-sectional regression this period (slope = that period's factor return)
    slope = np.cov(returns, exposures)[0, 1] / np.var(exposures)
    period_betas.append(slope)

period_betas = np.array(period_betas)

# Step 2: average the period estimates and test significance
avg_premium = period_betas.mean()
t_stat_fm, p_value_fm = stats.ttest_1samp(period_betas, popmean=0)

print(f"Average estimated factor premium: {avg_premium:.5f}")
print(f"t-stat: {t_stat_fm:.3f}, p-value: {p_value_fm:.4f}")


### Time series statistics: stationarity, autocorrelation, chronological validation

**Stationarity:** statistical properties (mean, variance) stay stable over time. Raw
price levels trend and aren't stationary; returns (differenced prices) generally are —
which is why modeling almost always happens on returns.

**Autocorrelation:** correlation of a series with a lagged version of itself. Strong
autocorrelation left in model residuals signals missing structure — good residuals
should look close to white noise.

**Chronological-only validation:** a random train/test split lets a model see "future"
data during training — direct look-ahead bias. Always split by time: train on the past,
test on what comes after.


In [ ]:
prices = 100 + np.cumsum(np.random.normal(0.05, 1, 250))     # simulated trending price series
returns = np.diff(prices) / prices[:-1]                       # differenced -> closer to stationary

print(f"Price series std across two halves: {prices[:125].std():.2f} vs {prices[125:].std():.2f}  (differs -> non-stationary)")
print(f"Return series std across two halves: {returns[:125].std():.4f} vs {returns[125:].std():.4f}  (closer -> more stationary)")


In [ ]:
# Chronological train/test split — the correct way to validate a time series model
split_point = int(len(returns) * 0.8)
train, test = returns[:split_point], returns[split_point:]
print(f"Train size: {len(train)}, Test size: {len(test)} — train is strictly earlier in time than test")


### Risk-adjusted return metrics


In [ ]:
risk_free_rate = 0.0001   # per-period

sharpe = (factor_returns.mean() - risk_free_rate) / factor_returns.std()

downside_returns = factor_returns[factor_returns < 0]
sortino = (factor_returns.mean() - risk_free_rate) / downside_returns.std()

print(f"Sharpe ratio: {sharpe:.3f}")
print(f"Sortino ratio: {sortino:.3f}")


**Why Sortino sometimes differs a lot from Sharpe:** Sharpe penalizes *all* volatility
equally, including upside surprises; Sortino only penalizes downside volatility — a
strategy with lumpy-but-positive returns can look worse under Sharpe than it "deserves"
relative to Sortino.


## Section 5 — Core Factors Used in Quantitative Finance

The main style factors that show up repeatedly in factor index construction. Each has
a typical metric, a rationale for why it earns a premium historically, and a common watch-out.


| Factor | Typical Metric(s) | Why It's Believed to Work | Watch-out |
|---|---|---|---|
| **Value** | P/E, P/B, EV/EBITDA | Cheap stocks relative to fundamentals tend to outperform over time (mean reversion / mispricing) | Can underperform for long stretches ("value trap" risk); negatively correlated with momentum |
| **Momentum** | Trailing 6-12 month return | Trends persist in the medium term due to underreaction/herding | Sharp reversals ("momentum crashes") especially after market stress |
| **Quality** | ROE, low debt/equity, earnings stability | Financially healthy companies are more resilient and consistently profitable | "Quality" is multi-dimensional — metric choice significantly changes results |
| **Size** | Market capitalization | Smaller companies have historically earned a premium, partly for illiquidity/risk | Premium has weakened/been inconsistent in recent decades |
| **Low Volatility** | Trailing return volatility, beta | Lower-vol stocks have historically delivered better risk-adjusted (not always absolute) returns than CAPM would predict | Can lag badly in strong bull markets; sector concentration risk (e.g., utilities) |
| **Yield** | Dividend yield | Income-focused investors reward consistent payers; often correlated with value | High yield can signal distress, not quality, if not paired with quality/payout-ratio checks |

**Cross-cutting themes worth being able to say out loud:**
- Value and momentum tend to be *negatively* correlated — combining them is a common
  diversification technique within factor portfolios.
- Factors are typically constructed **sector-neutral** (z-scored within sector) to avoid
  conflating a factor bet with an implicit sector bet.
- Factor premiums are cyclical — they go through extended periods of under- and
  out-performance, which is why single-period backtests are misleading (ties back to
  the hypothesis-testing and Fama-MacBeth material above).


## Section 6 — Expected Technical/Theory Questions & Good-Fit Answers

Background and "tell me about yourself" territory is skipped here — this section is
purely technical/theoretical, pulled from the material across all prior sections, in
the phrasing a director-level interviewer is likely to actually use.


**Q: "Why would you z-score a factor within its sector instead of across the whole universe?"**

A: Sectors trade at structurally different valuation levels — tech multiples aren't
comparable to utilities multiples — so a universe-wide z-score would mostly measure
which sector a stock is in, not whether it's genuinely cheap or expensive relative to
its true peer group. Sector-neutral z-scoring isolates the actual factor signal from
that sector-level noise, and it's also what keeps a factor bet from turning into an
unintended sector bet.


**Q: "Walk me through what beta and alpha actually represent, and how you'd calculate them."**

A: Beta measures a stock's sensitivity to market moves — covariance of the stock's
returns with the market's returns, divided by the variance of the market's returns.
Alpha is what's left over after removing what beta explains: the stock's average
return minus beta times the market's average return. In a factor context, alpha is
the part of performance you can actually attribute to the factor itself, not just to
broad market exposure — which is why isolating it matters before claiming a factor works.


**Q: "Why can't you just use a random train/test split for a time series model?"**

A: A random split lets the model train on data that comes chronologically after the
point it's being tested on — direct look-ahead bias, since the model gets information
it would never have in production. The fix is a chronological split: train only on the
past, test only on what follows, either with an expanding window or a rolling window
that both move forward in time.


**Q: "If your isolation forest is flagging 15% of securities as anomalies when you'd
expect 1-2%, what's your diagnostic path?"**

A: First I'd check the contamination parameter — that's the cheapest, most mechanical
explanation and the first thing I'd rule in or out. If that's not it, I'd look for
distributional shift or drift upstream, since a change in an input feature's scale or
range can make the model see far more of the data as extreme than it should. Only after
that would I sample the actual flagged points directly and check whether they look
genuinely anomalous or just borderline — I wouldn't reach for "overfitting" here, since
that concept doesn't cleanly apply to an unsupervised model with no labeled ground truth.


**Q: "Isolation Forest versus an autoencoder for anomaly detection — how would you choose?"**

A: Isolation Forest is my default first choice — it's cheaper to train, needs no
gradient tuning, and works well when anomalies are globally extreme or sit in sparse
regions of the feature space. I'd reach for an autoencoder when the anomalies are more
subtle or non-linear — cases where "abnormal" isn't simply "far from everything else"
but "doesn't fit the structure the model has learned" — accepting the added training
and tuning cost only when that complexity is actually needed.


**Q: "Why does Fama-MacBeth exist instead of just running one pooled regression across
all securities and periods?"**

A: A pooled regression treats every security-period observation as independent, but
returns across different securities in the same period are correlated with each other —
they're all reacting to the same market conditions that period. That correlation means
a pooled regression understates the real uncertainty in the factor return estimate.
Fama-MacBeth corrects for this by running a separate cross-sectional regression each
period, then testing whether the average of those period-by-period estimates is
significantly different from zero — which respects the time structure instead of
pretending every observation is independent.


**Q: "What's the practical difference between Sharpe and Sortino, and when would they
actually disagree?"**

A: Sharpe penalizes all volatility equally, including upside surprises; Sortino only
penalizes downside volatility. They diverge most for a strategy or factor with
lumpy-but-positive returns — something that has occasional big positive jumps alongside
smaller losses will look worse under Sharpe than it "deserves" to, because Sharpe treats
that upside volatility as a negative the same way it treats downside risk.


**Q: "Explain what a p-value actually tells you, and where people usually get it wrong."**

A: The p-value is the probability of observing data this extreme if the null hypothesis
were actually true — for a factor return test, that's the probability of seeing a return
this large purely by chance if the factor's true average return were zero. The common
mistake is reading it as "the probability the null hypothesis is true," which is a
different, and wrong, statement — the p-value never tells you the probability of a
hypothesis being true, only how surprising the data would be under an assumption.


**Q: "Why do you difference prices into returns before doing time series analysis on them?"**

A: Raw price levels trend and drift over time, which means their statistical
properties — mean, variance — aren't stable, so they're non-stationary. Most modeling
techniques implicitly assume some stability in what they're learning from, and modeling
a trending price series directly can make a model look like it's learned real structure
when it's actually just fit to a trend that won't repeat. Differencing prices into
returns removes that trend and produces a series that's much closer to stationary,
which is why nearly all time series work in finance happens on returns.


**Q: "Why is Polars faster than pandas, at a mechanical level, not just 'it's built in Rust'?"**

A: The real difference is lazy evaluation — Polars can build a full query plan before
executing anything, which lets it apply optimizations like predicate pushdown and
projection pushdown: pushing filters down to the data scan itself and only reading the
columns that are actually needed. Pandas executes each line eagerly with no visibility
into what comes next, so it can't make those same cross-line optimizations. The
Rust implementation matters too, but the query-plan optimization is the more
substantive answer.


**Q: "How would you validate a new factor before it's trusted in an autonomous,
zero-manual-review pipeline?"**

A: I'd start with the statistical significance of its returns — testing whether the
average return is distinguishable from zero, ideally with a Fama-MacBeth-style approach
rather than a single pooled regression, so the test respects the correlation across
securities in the same period. I'd also check it's been validated out-of-sample with
chronological, not random, splits to rule out look-ahead bias, and I'd want to see its
correlation with existing factors already in production — a high correlation with an
existing factor means it's not adding independent signal, just redundant exposure.


## Closing note

This notebook is meant to be run top-to-bottom once to confirm everything executes,
then used as a reference to re-derive answers out loud from the **Why** notes rather
than reading code verbatim in an interview setting.
